In [1]:
import cv2
import numpy as np
import os
from glob import glob

# ==============================
# CONFIG
# ==============================
INPUT_DIR = r"D:\projects\neuro\all fascicles\fascicle images\set2_Image_01_20x_bf_05_all"
OUTPUT_DIR = r"D:\projects\neuro\blurmaps\20x_bf_05_all"
WINDOW_SIZES = [7, 9, 11, 13]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================
# Function: local Laplacian variance
# ==============================
def local_laplacian_variance(lap_img, k):
    mean = cv2.blur(lap_img, (k, k))
    mean_sq = cv2.blur(lap_img ** 2, (k, k))
    var = mean_sq - mean ** 2
    var = cv2.normalize(var, None, 0, 255, cv2.NORM_MINMAX)
    return var.astype(np.uint8)

# ==============================
# Process all PNG images
# ==============================
png_files = glob(os.path.join(INPUT_DIR, "*.png"))

if len(png_files) == 0:
    raise RuntimeError("No PNG images found in input directory!")

for img_path in png_files:
    filename = os.path.splitext(os.path.basename(img_path))[0]
    print(f"Processing: {filename}.png")

    # Read image
    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ Could not read {img_path}, skipping.")
        continue

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Output subfolder per image
    img_out_dir = os.path.join(OUTPUT_DIR, filename)
    os.makedirs(img_out_dir, exist_ok=True)

    # --------------------------
    # Save Gaussian low-pass outputs
    # --------------------------
    # for k in WINDOW_SIZES:
    #     gaussian = cv2.GaussianBlur(gray, (k, k), 0)
    #     gauss_path = os.path.join(img_out_dir, f"gaussian_{k}x{k}.png")
    #     cv2.imwrite(gauss_path, gaussian)

    # --------------------------
    # Denoising for Laplacian
    # --------------------------
    gray_denoised = cv2.GaussianBlur(gray, (3, 3), 0)

    # Laplacian
    lap = cv2.Laplacian(gray_denoised, cv2.CV_64F)
    lap = np.abs(lap)

    # --------------------------
    # Blur maps (multi-scale)
    # --------------------------
    for k in WINDOW_SIZES:
        blur_map = local_laplacian_variance(lap, k)
        out_path = os.path.join(img_out_dir, f"blur_map_{k}x{k}.png")
        cv2.imwrite(out_path, blur_map)

print("✅ Done processing all PNG images.")

Processing: 1.png
Processing: 2.png
Processing: 3.png
Processing: 4.png
Processing: 5.png
✅ Done processing all PNG images.


In [4]:
import cv2
import numpy as np
import os
from glob import glob

# ==============================
# CONFIG
# ==============================
INPUT_DIR = r"D:\projects\neuro\all fascicles\fascicle images\set2_Image_01_20x_bf_05_all"
OUTPUT_DIR = r"D:\projects\neuro\blurmaps\20x_bf_05_all"

WINDOW_SIZES = [7, 9, 11, 13]
THRESH = 0.2   # 🔥 Tune this (0.2–0.5)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================
# Function: local Laplacian variance
# ==============================
def local_laplacian_variance(lap_img, k):
    mean = cv2.blur(lap_img, (k, k))
    mean_sq = cv2.blur(lap_img ** 2, (k, k))
    var = mean_sq - mean ** 2
    var = cv2.normalize(var, None, 0, 255, cv2.NORM_MINMAX)
    return var.astype(np.uint8)

# ==============================
# Process all PNG images
# ==============================
png_files = glob(os.path.join(INPUT_DIR, "*.png"))

if len(png_files) == 0:
    raise RuntimeError("No PNG images found in input directory!")

for img_path in png_files:
    filename = os.path.splitext(os.path.basename(img_path))[0]
    print(f"Processing: {filename}.png")

    # Read image
    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ Could not read {img_path}, skipping.")
        continue

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Output subfolder per image
    img_out_dir = os.path.join(OUTPUT_DIR, filename)
    os.makedirs(img_out_dir, exist_ok=True)

    # --------------------------
    # Denoising
    # --------------------------
    gray_denoised = cv2.GaussianBlur(gray, (3, 3), 0)

    # Laplacian
    lap = cv2.Laplacian(gray_denoised, cv2.CV_64F)
    lap = np.abs(lap)

    blur_maps = []

    # ==========================
    # MULTI-SCALE BLUR MAPS
    # ==========================
    for k in WINDOW_SIZES:
        blur_map = local_laplacian_variance(lap, k)
        blur_maps.append(blur_map)

        # Save blur map
        cv2.imwrite(os.path.join(img_out_dir, f"blur_map_{k}x{k}.png"), blur_map)

    # ==========================
    # COMBINE MULTI-SCALE (IMPORTANT)
    # ==========================
    combined_blur = np.maximum.reduce(blur_maps)

    cv2.imwrite(os.path.join(img_out_dir, "blur_map_combined.png"), combined_blur)

    # ==========================
    # NORMALIZE
    # ==========================
    blur_norm = combined_blur.astype(np.float32) / 255.0

    # ==========================
    # THRESHOLD → SHARP REGIONS
    # ==========================
    sharp_mask = (blur_norm > THRESH).astype(np.uint8)

    # ==========================
    # CLEAN MASK (VERY IMPORTANT)
    # ==========================
    kernel = np.ones((3, 3), np.uint8)
    sharp_mask = cv2.morphologyEx(sharp_mask, cv2.MORPH_OPEN, kernel)
    sharp_mask = cv2.morphologyEx(sharp_mask, cv2.MORPH_DILATE, kernel)

    # ==========================
    # APPLY MASK
    # ==========================
    sharp_only = cv2.bitwise_and(img, img, mask=sharp_mask)

    # ==========================
    # SAVE OUTPUTS
    # ==========================
    cv2.imwrite(os.path.join(img_out_dir, "sharp_mask.png"), sharp_mask * 255)
    cv2.imwrite(os.path.join(img_out_dir, "sharp_only.png"), sharp_only)

print("✅ Done processing all PNG images.")

Processing: 1.png
Processing: 2.png
Processing: 3.png
Processing: 4.png
Processing: 5.png
✅ Done processing all PNG images.


In [10]:
import cv2
import numpy as np
import os
from glob import glob

# ==============================
# CONFIG
# ==============================
INPUT_DIR = r"D:\projects\neuro\all fascicles\fascicle images\set2_Image_01_20x_bf_05_all"
OUTPUT_DIR = r"D:\projects\neuro\blurmaps\20x_bf_05_all_3"
WINDOW_SIZES = [7, 9, 11, 13]
BLUR_THRESH = 0.15
MIN_AREA = 5000

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================
def local_laplacian_variance(lap_img, k):
    mean = cv2.blur(lap_img, (k, k))
    mean_sq = cv2.blur(lap_img ** 2, (k, k))
    var = mean_sq - mean ** 2
    return var.astype(np.float32)

# ==============================
png_files = glob(os.path.join(INPUT_DIR, "*.png"))

for img_path in png_files:
    filename = os.path.splitext(os.path.basename(img_path))[0]
    print(f"Processing: {filename}.png")

    img = cv2.imread(img_path)
    if img is None:
        print(f"  ⚠️  Could not read {filename}, skipping.")
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # ==========================
    # TISSUE MASK (exclude black background)
    # ==========================
    tissue_mask = (gray > 10).astype(np.uint8)
    tissue_mask = cv2.morphologyEx(
        tissue_mask, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8)
    )

    # ==========================
    # STROMA MASK (pink regions — low texture but NOT blurry)
    # protect these from ever being removed
    # ==========================
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    # pink: hue ~150-180 or 0-10, low-medium saturation, high value
    pink_lo1 = cv2.inRange(hsv, (150, 15, 140), (180, 160, 255))
    pink_lo2 = cv2.inRange(hsv, (0,  15, 140), (10,  160, 255))
    stroma_mask = cv2.bitwise_or(pink_lo1, pink_lo2)
    # dilate slightly to cover stroma edges
    stroma_mask = cv2.morphologyEx(
        stroma_mask, cv2.MORPH_DILATE, np.ones((7, 7), np.uint8)
    )
    # only within tissue
    stroma_mask = cv2.bitwise_and(stroma_mask, tissue_mask)

    # ==========================
    # LAPLACIAN
    # ==========================
    gray_blur = cv2.GaussianBlur(gray, (3, 3), 0)
    lap = np.abs(cv2.Laplacian(gray_blur, cv2.CV_64F))

    # ==========================
    # MULTI-SCALE LOCAL VARIANCE
    # ==========================
    blur_maps = [local_laplacian_variance(lap, k) for k in WINDOW_SIZES]
    combined_blur = np.maximum.reduce(blur_maps)

    # ==========================
    # NORMALIZE WITHIN TISSUE
    # ==========================
    tissue_pixels = combined_blur[tissue_mask == 1]
    p99 = np.percentile(tissue_pixels, 99) + 1e-6
    blur_norm = np.clip(combined_blur / p99, 0, 1).astype(np.float32)

    blur_map_u8 = (blur_norm * 255).astype(np.uint8)
    cv2.imwrite(os.path.join(OUTPUT_DIR, filename + "_blur.png"), blur_map_u8)

    # ==========================
    # BLURRY MASK — inside tissue, NOT stroma
    # stroma is low variance but sharp, so exclude it here
    # ==========================
    blur_mask = (
        (blur_norm < BLUR_THRESH) &
        (tissue_mask == 1) &
        (stroma_mask == 0)          # ← key fix
    ).astype(np.uint8)

    # ==========================
    # LARGE BLURRY BLOBS ONLY
    # ==========================
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        blur_mask, connectivity=8
    )

    remove_mask = np.zeros_like(blur_mask)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] > MIN_AREA:
            remove_mask[labels == i] = 1

    # ==========================
    # KEEP = tissue AND NOT removed
    # ==========================
    keep_mask = cv2.bitwise_and(
        tissue_mask,
        (1 - remove_mask).astype(np.uint8)
    )
    keep_mask = cv2.morphologyEx(keep_mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))

    # ==========================
    # OUTPUTS
    # ==========================
    sharp_only = cv2.bitwise_and(img, img, mask=keep_mask)

    blur_norm_3ch = cv2.merge([blur_norm] * 3)
    img_float = img.astype(np.float32) / 255.0
    weighted = img_float * (0.5 + 0.5 * blur_norm_3ch)
    weighted[tissue_mask == 0] = 0
    weighted = (weighted * 255).astype(np.uint8)

    # debug: blurry zones = red tint, stroma = green tint
    overlay = img.copy()
    overlay[remove_mask == 1] = (
        overlay[remove_mask == 1] * 0.4 + np.array([0, 0, 180]) * 0.6
    ).astype(np.uint8)
    overlay[stroma_mask == 1] = (
        overlay[stroma_mask == 1] * 0.6 + np.array([0, 180, 0]) * 0.4
    ).astype(np.uint8)
    overlay[tissue_mask == 0] = 0

    cv2.imwrite(os.path.join(OUTPUT_DIR, filename + "_keep_mask.png"), keep_mask * 255)
    cv2.imwrite(os.path.join(OUTPUT_DIR, filename + "_sharp_only.png"), sharp_only)
    cv2.imwrite(os.path.join(OUTPUT_DIR, filename + "_weighted.png"), weighted)
    cv2.imwrite(os.path.join(OUTPUT_DIR, filename + "_debug_overlay.png"), overlay)

    total_tissue = np.sum(tissue_mask)
    removed = np.sum(remove_mask)
    pct = 100 * removed / (total_tissue + 1e-6)
    print(f"  tissue px: {total_tissue} | removed px: {removed} ({pct:.1f}%)")

print("✅ Done")

Processing: 1.png
  tissue px: 667088 | removed px: 0 (0.0%)
Processing: 2.png
  tissue px: 1565125 | removed px: 67822 (4.3%)
Processing: 3.png
  tissue px: 2208488 | removed px: 6563 (0.3%)
Processing: 4.png
  tissue px: 1684798 | removed px: 22992 (1.4%)
Processing: 5.png
  tissue px: 1323324 | removed px: 179234 (13.5%)
✅ Done


In [5]:
import cv2
import numpy as np
import os
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# --------------------------
# CONFIG
# --------------------------
input_folder = r"D:\projects\neuro\all fascicles\fascicle images\set2_Image_01_20x_bf_05_all"
output_root  = r"C:\Users\DELL\Downloads\detetction"
os.makedirs(output_root, exist_ok=True)

scale_factor = 0.136

N_RAYS   = 36
RAY_STEP = 0.5

TISSUE_OVERLAP_MIN = 0.50
MAX_MEAN_INTENSITY = 245
MIN_CIRCULARITY    = 0.05
MIN_CONTOUR_AREA   = 8
MAX_CONTOUR_AREA   = 2000000
MIN_SOLIDITY       = 0.15

BLUR_WINDOW_SIZES  = [7, 9, 11, 13]
BLUR_THRESH        = 0.15
BLUR_MIN_AREA      = 5000
STROMA_SHARP_MIN   = 0.05

RAY_ANGLES           = np.linspace(0, 360, N_RAYS, endpoint=False)
OUTER_RAY_COL_LABELS = [f"outer_ray_{int(a):03d}deg_um"     for a in RAY_ANGLES]
INNER_RAY_COL_LABELS = [f"inner_ray_{int(a):03d}deg_um"     for a in RAY_ANGLES]
THICK_RAY_COL_LABELS = [f"thickness_ray_{int(a):03d}deg_um" for a in RAY_ANGLES]


# --------------------------
# Helpers
# --------------------------
def darken_and_sharpen(image):
    darkened = cv2.convertScaleAbs(image, alpha=1.5, beta=-20)
    sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    return cv2.filter2D(darkened, -1, sharpen_kernel)


def build_tissue_mask(full_image):
    hsv = cv2.cvtColor(full_image, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(hsv)
    sat_mask       = cv2.inRange(S, 15, 255)
    not_too_bright = cv2.inRange(V, 0, 250)
    mask = cv2.bitwise_and(sat_mask, not_too_bright)
    k    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    return mask


def local_laplacian_variance(lap_img, k):
    mean    = cv2.blur(lap_img, (k, k))
    mean_sq = cv2.blur(lap_img ** 2, (k, k))
    return (mean_sq - mean ** 2).astype(np.float32)


def build_sharp_mask(image, tissue_mask):
    gray   = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    hsv    = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    pink1  = cv2.inRange(hsv, (145, 10, 120), (180, 200, 255))
    pink2  = cv2.inRange(hsv, (  0, 10, 120), ( 15, 200, 255))
    stroma = cv2.bitwise_or(pink1, pink2)
    stroma = cv2.morphologyEx(stroma, cv2.MORPH_DILATE, np.ones((7, 7), np.uint8))
    stroma = cv2.bitwise_and(stroma, tissue_mask)

    gray_b   = cv2.GaussianBlur(gray, (3, 3), 0)
    lap      = np.abs(cv2.Laplacian(gray_b, cv2.CV_64F))
    maps     = [local_laplacian_variance(lap, k) for k in BLUR_WINDOW_SIZES]
    combined = np.maximum.reduce(maps)

    tp        = combined[tissue_mask == 1]
    p99       = (np.percentile(tp, 99) if len(tp) else 1.0) + 1e-6
    blur_norm = np.clip(combined / p99, 0, 1).astype(np.float32)

    stroma_sharp = (blur_norm > STROMA_SHARP_MIN).astype(np.uint8)
    stroma       = cv2.bitwise_and(stroma, stroma_sharp)

    blur_mask = (
        (blur_norm < BLUR_THRESH) &
        (tissue_mask == 1) &
        (stroma == 0)
    ).astype(np.uint8)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(blur_mask, connectivity=8)
    remove_mask = np.zeros_like(blur_mask)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] > BLUR_MIN_AREA:
            remove_mask[labels == i] = 1

    sharp_mask = cv2.bitwise_and(tissue_mask, (1 - remove_mask).astype(np.uint8))
    sharp_mask = cv2.morphologyEx(sharp_mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    return sharp_mask * 255


def contour_overlap_ratio(contour, mask):
    c_mask = np.zeros(mask.shape, dtype=np.uint8)
    cv2.drawContours(c_mask, [contour], -1, 255, -1)
    inter  = cv2.bitwise_and(c_mask, mask)
    area_c = cv2.countNonZero(c_mask)
    area_i = cv2.countNonZero(inter)
    return area_i / float(area_c) if area_c > 0 else 0.0


def circularity(contour):
    a = cv2.contourArea(contour)
    p = cv2.arcLength(contour, True)
    return 4 * np.pi * a / (p * p + 1e-6) if p > 0 else 0


# --------------------------
# Ray casting
# --------------------------
def cast_rays_to_boundary(contour, cx, cy, n_rays=N_RAYS, step=RAY_STEP):
    x_b, y_b, w_b, h_b = cv2.boundingRect(contour)
    pad      = 20
    max_r    = int(max(w_b, h_b) + pad)
    local_cx = max_r
    local_cy = max_r
    shifted  = contour - np.array([[int(cx) - max_r, int(cy) - max_r]])

    mask_size  = max_r * 2 + 2
    local_mask = np.zeros((mask_size, mask_size), dtype=np.uint8)
    cv2.drawContours(local_mask, [shifted], -1, 255, -1)

    angles           = np.linspace(0, 2 * np.pi, n_rays, endpoint=False)
    ray_distances_px = np.zeros(n_rays)

    for i, angle in enumerate(angles):
        cos_a  = np.cos(angle)
        sin_a  = np.sin(angle)
        r_vals = np.arange(0, max_r, step)
        xs     = (local_cx + r_vals * cos_a).astype(int)
        ys     = (local_cy + r_vals * sin_a).astype(int)
        valid  = (xs >= 0) & (xs < mask_size) & (ys >= 0) & (ys < mask_size)
        xs_v, ys_v, r_v = xs[valid], ys[valid], r_vals[valid]

        if len(xs_v) == 0:
            continue

        pixel_vals  = local_mask[ys_v, xs_v]
        inside_idx  = np.where(pixel_vals == 255)[0]
        if len(inside_idx) == 0:
            continue

        outside_after = np.where(
            (pixel_vals == 0) & (np.arange(len(pixel_vals)) > inside_idx[0])
        )[0]
        ray_distances_px[i] = r_v[outside_after[0]] if len(outside_after) else r_v[-1]

    return ray_distances_px, np.degrees(angles)


def draw_rays_on_image(output_image, cx, cy, ray_distances_px,
                       ray_angles_deg, chosen_idx, x_offset, y_offset):
    angles_rad = np.deg2rad(ray_angles_deg)
    for i, (r, ang) in enumerate(zip(ray_distances_px, angles_rad)):
        end_x = int(cx + r * np.cos(ang))
        end_y = int(cy + r * np.sin(ang))
        color = (0, 0, 255) if i == chosen_idx else (200, 200, 0)
        cv2.line(output_image,
                 (int(cx + x_offset), int(cy + y_offset)),
                 (end_x + x_offset,   end_y + y_offset),
                 color, 1)



def save_before_after(image, sharp_mask_full, output_image, img_output_folder, img_name):
    # BEFORE: original image with blurry regions darkened/greyed out
    before = image.copy()
    blurry_zone = (sharp_mask_full == 0)
    # grey out blurry regions so it's visually clear what's being ignored
    before[blurry_zone] = (before[blurry_zone] * 0.25).astype(np.uint8)
    # draw a red border around blurry zones
    blur_vis = ((sharp_mask_full == 0) & 
                (cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) > 10)).astype(np.uint8)
    contours_blur, _ = cv2.findContours(blur_vis, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(before, contours_blur, -1, (0, 0, 255), 2)

    # AFTER: the annotated output (already drawn with detections)
    after = output_image.copy()
    # also grey out blurry regions on after image so comparison is clean
    after[blurry_zone] = (after[blurry_zone] * 0.25).astype(np.uint8)

    cv2.imwrite(os.path.join(img_output_folder, f"before_{img_name}"), before)
    cv2.imwrite(os.path.join(img_output_folder, f"after_{img_name}"),  after)
    print(f"  ✔ Before/after images saved → {img_output_folder}")
# --------------------------
# Patch processing
# --------------------------
def process_patch(patch, mask_patch, sharp_patch,
                  x_offset, y_offset,
                  axon_data, object_counter,
                  output_image, axons_image):

    patch    = darken_and_sharpen(patch)
    gray     = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    smooth   = cv2.bilateralFilter(gray, 5, 20, 20)
    clahe    = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(smooth)
    blurred  = cv2.GaussianBlur(enhanced, (3, 3), 0)

    _, otsu  = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    adaptive = cv2.adaptiveThreshold(blurred, 255,
                                     cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY_INV, 35, 2)
    thresh   = cv2.bitwise_or(otsu, adaptive)
    thresh   = cv2.bitwise_and(thresh, thresh, mask=sharp_patch)

    kernel  = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    sure_bg = cv2.dilate(opening, kernel, iterations=3)

    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    if dist_transform.max() > 0:
        _, sure_fg = cv2.threshold(dist_transform, 0.4 * dist_transform.max(), 255, 0)
    else:
        sure_fg = np.zeros_like(opening)
    sure_fg = np.uint8(sure_fg)

    unknown = cv2.subtract(sure_bg, sure_fg)
    num_markers, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0

    wshed_input = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    markers     = cv2.watershed(wshed_input, markers)

    cleaned = thresh.copy()
    cleaned[markers == -1] = 0

    contours, hierarchy = cv2.findContours(cleaned, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    if hierarchy is None:
        return object_counter
    hierarchy = hierarchy[0]

    for i, contour in enumerate(contours):
        if hierarchy[i][3] != -1:
            continue

        area = cv2.contourArea(contour)
        if area < MIN_CONTOUR_AREA or area > MAX_CONTOUR_AREA:
            continue

        M = cv2.moments(contour)
        if M["m00"] > 0:
            cx_c = int(M["m10"] / M["m00"])
            cy_c = int(M["m01"] / M["m00"])
            if sharp_patch[cy_c, cx_c] == 0:
                continue

        overlap = contour_overlap_ratio(contour, mask_patch)
        if overlap < TISSUE_OVERLAP_MIN:
            continue

        hull     = cv2.convexHull(contour)
        solidity = area / (cv2.contourArea(hull) + 1e-6)
        circ     = circularity(contour)
        if solidity < MIN_SOLIDITY or circ < MIN_CIRCULARITY:
            continue

        inner_children = [
            contours[j] for j in range(len(contours))
            if hierarchy[j][3] == i and cv2.contourArea(contours[j]) > 30
        ]
        if len(inner_children) == 0:
            continue

        axon_type = "mature" if len(inner_children) <= 2 else "regrowth_cluster"

        (x_outer, y_outer), outer_radius_enc = cv2.minEnclosingCircle(contour)
        contour_shifted = contour + np.array([x_offset, y_offset])
        x_outer_abs     = x_outer + x_offset
        y_outer_abs     = y_outer + y_offset

        outer_area_px  = cv2.contourArea(contour)
        outer_area_um2 = outer_area_px * (scale_factor ** 2)
        outer_color    = (255, 0, 0) if axon_type == "mature" else (0, 255, 255)

        cv2.drawContours(output_image, [contour_shifted], -1, outer_color, 1)
        cv2.drawContours(axons_image,  [contour_shifted], -1, outer_color, 1)

        inner_radii, inner_areas = [], []
        for inner_c in inner_children:
            (_, _), ir = cv2.minEnclosingCircle(inner_c)
            inner_shifted = inner_c + np.array([x_offset, y_offset])
            cv2.drawContours(output_image, [inner_shifted], -1, (0, 255, 0), 1)
            cv2.drawContours(axons_image,  [inner_shifted], -1, (0, 255, 0), 1)
            inner_radii.append(ir)
            inner_areas.append(cv2.contourArea(inner_c))

        inner_radius   = max(inner_radii)
        inner_area_px  = max(inner_areas)
        inner_area_um2 = inner_area_px * (scale_factor ** 2)

        outer_ray_dict = {col: np.nan for col in OUTER_RAY_COL_LABELS}
        inner_ray_dict = {col: np.nan for col in INNER_RAY_COL_LABELS}
        thick_ray_dict = {col: np.nan for col in THICK_RAY_COL_LABELS}

        outer_radius_um  = outer_radius_enc * scale_factor
        inner_radius_um  = inner_radius     * scale_factor
        chosen_outer_um  = outer_radius_um
        chosen_inner_um  = inner_radius_um
        chosen_thick_um  = max(outer_radius_um - inner_radius_um, 0.0)
        chosen_ray_angle = np.nan
        chosen_idx       = -1

        if axon_type == "mature":
            outer_rays_px, ray_angles_deg = cast_rays_to_boundary(contour, x_outer, y_outer)
            outer_rays_um = outer_rays_px * scale_factor

            largest_inner_c  = inner_children[np.argmax(inner_areas)]
            inner_rays_px, _ = cast_rays_to_boundary(largest_inner_c, x_outer, y_outer)
            inner_rays_um    = inner_rays_px * scale_factor
            thick_per_dir    = outer_rays_um - inner_rays_um

            for o_col, i_col, t_col, o_val, i_val, t_val in zip(
                OUTER_RAY_COL_LABELS, INNER_RAY_COL_LABELS, THICK_RAY_COL_LABELS,
                outer_rays_um, inner_rays_um, thick_per_dir
            ):
                outer_ray_dict[o_col] = round(float(o_val), 4)
                inner_ray_dict[i_col] = round(float(i_val), 4)
                thick_ray_dict[t_col] = round(float(t_val), 4)

            positive_mask = thick_per_dir > 0
            if positive_mask.any():
                valid_thick      = np.where(positive_mask, thick_per_dir, np.inf)
                chosen_idx       = int(np.argmin(valid_thick))
                chosen_outer_um  = float(outer_rays_um[chosen_idx])
                chosen_inner_um  = float(inner_rays_um[chosen_idx])
                chosen_thick_um  = float(thick_per_dir[chosen_idx])
                chosen_ray_angle = float(ray_angles_deg[chosen_idx])
            else:
                chosen_thick_um  = max(outer_radius_um - inner_radius_um, 0.0)

            draw_rays_on_image(output_image, x_outer, y_outer,
                               outer_rays_px, ray_angles_deg,
                               chosen_idx, x_offset, y_offset)

        thickness_um = chosen_thick_um
        g_ratio      = chosen_inner_um / (chosen_outer_um + 1e-6)
        area_ratio   = inner_area_um2  / (outer_area_um2  + 1e-6)

        record = {
            "axon_id"              : object_counter,
            "axon_type"            : axon_type,
            "num_inner_contours"   : len(inner_children),
            "center_x_px"          : round(float(x_outer_abs), 2),
            "center_y_px"          : round(float(y_outer_abs), 2),
            "outer_radius_enc_um"  : round(outer_radius_um,  4),
            "inner_radius_enc_um"  : round(inner_radius_um,  4),
            "chosen_ray_angle_deg" : round(chosen_ray_angle, 2) if axon_type == "mature" else np.nan,
            "chosen_outer_ray_um"  : round(chosen_outer_um,  4),
            "chosen_inner_ray_um"  : round(chosen_inner_um,  4),
            "thickness_um"         : round(thickness_um,     4),
            "diameter_um"          : round(2 * chosen_outer_um, 4),
            "outer_area_um2"       : round(outer_area_um2,   4),
            "inner_area_um2"       : round(inner_area_um2,   4),
            "area_ratio"           : round(area_ratio,        4),
            "g_ratio"              : round(g_ratio,           4),
        }
        record.update(outer_ray_dict)
        record.update(inner_ray_dict)
        record.update(thick_ray_dict)
        axon_data.append(record)

        cv2.putText(output_image, str(object_counter),
                    (int(x_outer_abs), int(y_outer_abs)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        cv2.putText(axons_image, str(object_counter),
                    (int(x_outer_abs), int(y_outer_abs)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        object_counter += 1

    return object_counter


# --------------------------
# Whole-image processing
# --------------------------
def process_image(image_path, patch_size=1024):
    image = cv2.imread(image_path)
    if image is None:
        print(f"❌ Cannot read: {image_path}")
        return None, None, None

    tissue_mask_full = build_tissue_mask(image)
    sharp_mask_full  = build_sharp_mask(image, tissue_mask_full)

    h, w = image.shape[:2]

    # AFTER — detections with blur regions ignored (existing behaviour)
    output_image   = image.copy()
    axons_after    = image.copy()
    axon_data      = []
    object_counter = 1

    for y in range(0, h, patch_size):
        for x in range(0, w, patch_size):
            patch       = image[y:y + patch_size, x:x + patch_size]
            mask_patch  = tissue_mask_full[y:y + patch_size, x:x + patch_size]
            sharp_patch = sharp_mask_full[ y:y + patch_size, x:x + patch_size]
            object_counter = process_patch(
                patch, mask_patch, sharp_patch,
                x, y, axon_data, object_counter,
                output_image, axons_after
            )

    # BEFORE — detections everywhere, no blur filtering
    axons_before      = image.copy()
    output_before     = image.copy()
    axon_data_before  = []
    object_counter_b  = 1
    full_sharp        = np.full((h, w), 255, dtype=np.uint8)  # no blur mask

    for y in range(0, h, patch_size):
        for x in range(0, w, patch_size):
            patch      = image[y:y + patch_size, x:x + patch_size]
            mask_patch = tissue_mask_full[y:y + patch_size, x:x + patch_size]
            sharp_patch_full = full_sharp[y:y + patch_size, x:x + patch_size]
            object_counter_b = process_patch(
                patch, mask_patch, sharp_patch_full,
                x, y, axon_data_before, object_counter_b,
                output_before, axons_before
            )

    print(f"  → WITH blur filter:    {len(axon_data)} axons")
    print(f"  → WITHOUT blur filter: {len(axon_data_before)} axons")
    return axon_data, output_image, axons_after, axons_before


# --------------------------
# Excel export
# --------------------------
def save_to_excel(axon_data, output_folder, image_name):
    if not axon_data:
        print("  ⚠ No axon data to save.")
        return None

    df = pd.DataFrame(axon_data)

    base_cols = [
        "axon_id", "axon_type", "num_inner_contours",
        "center_x_px", "center_y_px",
        "outer_radius_enc_um", "inner_radius_enc_um",
        *OUTER_RAY_COL_LABELS, *INNER_RAY_COL_LABELS, *THICK_RAY_COL_LABELS,
        "chosen_ray_angle_deg", "chosen_outer_ray_um", "chosen_inner_ray_um",
        "thickness_um", "diameter_um",
        "outer_area_um2", "inner_area_um2",
        "area_ratio", "g_ratio",
    ]
    ordered_cols = [c for c in base_cols if c in df.columns]
    df = df[ordered_cols]

    excel_path = os.path.join(output_folder, f"{image_name}_axon_measurements.xlsx")
    df.to_excel(excel_path, index=False, sheet_name="Axon Measurements")

    wb = openpyxl.load_workbook(excel_path)
    ws = wb.active

    HEADER_FILL    = PatternFill("solid", fgColor="1F4E79")
    OUTER_RAY_FILL = PatternFill("solid", fgColor="E8F4FD")
    INNER_RAY_FILL = PatternFill("solid", fgColor="EAF4E8")
    THICK_RAY_FILL = PatternFill("solid", fgColor="FFF2CC")
    MATURE_FILL    = PatternFill("solid", fgColor="D9F0D3")
    REGROWTH_FILL  = PatternFill("solid", fgColor="FCE4D6")
    CHOSEN_FILL    = PatternFill("solid", fgColor="FFD700")
    THICK_FILL     = PatternFill("solid", fgColor="F4CCCC")
    WHITE_FONT     = Font(color="FFFFFF", bold=True, name="Calibri", size=10)
    HEADER_BORDER  = Border(bottom=Side(style="medium", color="FFFFFF"))
    CELL_BORDER    = Border(bottom=Side(style="thin",   color="D0D0D0"))

    col_names = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]

    def col_idx(name):
        try:    return col_names.index(name) + 1
        except: return None

    outer_ray_col_indices = [col_idx(c) for c in OUTER_RAY_COL_LABELS if col_idx(c)]
    inner_ray_col_indices = [col_idx(c) for c in INNER_RAY_COL_LABELS if col_idx(c)]
    thick_ray_col_indices = [col_idx(c) for c in THICK_RAY_COL_LABELS if col_idx(c)]
    chosen_cols           = {col_idx(c) for c in
                             ("chosen_ray_angle_deg", "chosen_outer_ray_um", "chosen_inner_ray_um")
                             if col_idx(c)}
    thickness_col  = col_idx("thickness_um")
    axon_type_col  = col_idx("axon_type")

    for cell in ws[1]:
        cell.fill      = HEADER_FILL
        cell.font      = WHITE_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = HEADER_BORDER

    for ci in outer_ray_col_indices:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="1565C0")
    for ci in inner_ray_col_indices:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="2E7D32")
    for ci in thick_ray_col_indices:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="F57F17")
    for ci in chosen_cols:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="B8860B")
    if thickness_col:
        ws.cell(1, thickness_col).fill = PatternFill("solid", fgColor="8B0000")
        ws.cell(1, thickness_col).font = Font(color="FFFFFF", bold=True, name="Calibri", size=10)

    for row_idx in range(2, ws.max_row + 1):
        atype    = ws.cell(row_idx, axon_type_col).value if axon_type_col else ""
        row_fill = MATURE_FILL if atype == "mature" else REGROWTH_FILL

        for col_idx_ in range(1, ws.max_column + 1):
            cell           = ws.cell(row_idx, col_idx_)
            cell.border    = CELL_BORDER
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.fill      = row_fill

            if   col_idx_ in outer_ray_col_indices: cell.fill = OUTER_RAY_FILL
            elif col_idx_ in inner_ray_col_indices: cell.fill = INNER_RAY_FILL
            elif col_idx_ in thick_ray_col_indices: cell.fill = THICK_RAY_FILL
            elif col_idx_ in chosen_cols:
                cell.fill = CHOSEN_FILL
                cell.font = Font(bold=True, name="Calibri", size=10)
            elif thickness_col and col_idx_ == thickness_col:
                cell.fill = THICK_FILL
                cell.font = Font(bold=True, name="Calibri", size=10)

    for col_num in range(1, ws.max_column + 1):
        col_letter = get_column_letter(col_num)
        header_val = str(ws.cell(1, col_num).value or "")
        if header_val.startswith(("outer_ray_", "inner_ray_", "thickness_ray_")):
            ws.column_dimensions[col_letter].width = 14
        elif header_val in ("axon_id", "axon_type", "num_inner_contours"):
            ws.column_dimensions[col_letter].width = 16
        else:
            ws.column_dimensions[col_letter].width = 20

    ws.row_dimensions[1].height = 40
    ws.freeze_panes = "A2"

    ws_summary  = wb.create_sheet("Summary")
    mature_df   = df[df["axon_type"] == "mature"]
    regrowth_df = df[df["axon_type"] == "regrowth_cluster"]

    summary_rows = [
        ("Image",                       image_name),
        ("Total axons",                 len(df)),
        ("Mature axons",                len(mature_df)),
        ("Regrowth clusters",           len(regrowth_df)),
        ("", ""),
        ("--- Mature Axons ---", ""),
        ("Mean outer radius enc. (µm)", round(mature_df["outer_radius_enc_um"].mean(), 4) if len(mature_df) else "N/A"),
        ("Mean inner radius enc. (µm)", round(mature_df["inner_radius_enc_um"].mean(), 4) if len(mature_df) else "N/A"),
        ("Mean chosen outer ray (µm)",  round(mature_df["chosen_outer_ray_um"].mean(),  4) if len(mature_df) else "N/A"),
        ("Mean chosen inner ray (µm)",  round(mature_df["chosen_inner_ray_um"].mean(),  4) if len(mature_df) else "N/A"),
        ("Mean thickness (µm)",         round(mature_df["thickness_um"].mean(),          4) if len(mature_df) else "N/A"),
        ("Mean g-ratio",                round(mature_df["g_ratio"].mean(),               4) if len(mature_df) else "N/A"),
        ("Mean diameter (µm)",          round(mature_df["diameter_um"].mean(),           4) if len(mature_df) else "N/A"),
        ("", ""),
        ("--- Regrowth Clusters ---", ""),
        ("Mean outer radius enc. (µm)", round(regrowth_df["outer_radius_enc_um"].mean(), 4) if len(regrowth_df) else "N/A"),
        ("Mean thickness (µm)",         round(regrowth_df["thickness_um"].mean(),        4) if len(regrowth_df) else "N/A"),
    ]

    for r, (label, value) in enumerate(summary_rows, start=1):
        c1 = ws_summary.cell(r, 1, value=label)
        c2 = ws_summary.cell(r, 2, value=value)
        c1.font      = Font(bold=True, name="Calibri", size=10)
        c2.alignment = Alignment(horizontal="center")
        if str(label).startswith("---"):
            c1.fill = HEADER_FILL
            c1.font = WHITE_FONT

    ws_summary.column_dimensions["A"].width = 38
    ws_summary.column_dimensions["B"].width = 20
    wb.save(excel_path)
    print(f"  ✔ Excel saved → {excel_path}")
    return df


# --------------------------
# Process folder
# --------------------------
def process_folder(input_folder, output_root):
    images = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
    ]
    if not images:
        print("⚠ No images found.")
        return

    for img_name in images:
        print(f"\n{'='*30}\n Processing: {img_name}\n{'='*30}")
        img_path          = os.path.join(input_folder, img_name)
        img_name_wo_ext   = os.path.splitext(img_name)[0]
        img_output_folder = os.path.join(output_root, img_name_wo_ext)
        os.makedirs(img_output_folder, exist_ok=True)

        axon_data, output_image, axons_after, axons_before = process_image(img_path)
        if axon_data is None:
            continue

        save_to_excel(axon_data, img_output_folder, img_name_wo_ext)

        cv2.imwrite(os.path.join(img_output_folder, f"numbered_{img_name}"),      output_image)
        cv2.imwrite(os.path.join(img_output_folder, f"axons_after_{img_name}"),   axons_after)   # with blur filter
        cv2.imwrite(os.path.join(img_output_folder, f"axons_before_{img_name}"),  axons_before)  # without blur filter
        print(f"  ✔ Images saved → {img_output_folder}")

    print("\n🎉 Done.")


process_folder(input_folder, output_root)


 Processing: 1.png
  → WITH blur filter:    191 axons
  → WITHOUT blur filter: 192 axons
  ✔ Excel saved → C:\Users\DELL\Downloads\detetction\1\1_axon_measurements.xlsx
  ✔ Images saved → C:\Users\DELL\Downloads\detetction\1

 Processing: 2.png
  → WITH blur filter:    333 axons
  → WITHOUT blur filter: 340 axons
  ✔ Excel saved → C:\Users\DELL\Downloads\detetction\2\2_axon_measurements.xlsx
  ✔ Images saved → C:\Users\DELL\Downloads\detetction\2

 Processing: 3.png
  → WITH blur filter:    526 axons
  → WITHOUT blur filter: 535 axons
  ✔ Excel saved → C:\Users\DELL\Downloads\detetction\3\3_axon_measurements.xlsx
  ✔ Images saved → C:\Users\DELL\Downloads\detetction\3

 Processing: 4.png
  → WITH blur filter:    426 axons
  → WITHOUT blur filter: 432 axons
  ✔ Excel saved → C:\Users\DELL\Downloads\detetction\4\4_axon_measurements.xlsx
  ✔ Images saved → C:\Users\DELL\Downloads\detetction\4

 Processing: 5.png
  → WITH blur filter:    273 axons
  → WITHOUT blur filter: 277 axons
  ✔ Ex

In [9]:
import cv2
import numpy as np
import os
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# --------------------------
# CONFIG
# --------------------------
input_folder = r"D:\projects\neuro\all fascicles\fascicle images\set2_Image_01_20x_bf_02"
output_root  = r"C:\Users\DELL\Downloads\20x_bf_02_detection"
os.makedirs(output_root, exist_ok=True)

scale_factor = 0.136

N_RAYS   = 36
RAY_STEP = 0.5

TISSUE_OVERLAP_MIN = 0.50
MAX_MEAN_INTENSITY = 245
MIN_CIRCULARITY    = 0.05
MIN_CONTOUR_AREA   = 8
MAX_CONTOUR_AREA   = 2000000
MIN_SOLIDITY       = 0.15

RAY_ANGLES           = np.linspace(0, 360, N_RAYS, endpoint=False)
OUTER_RAY_COL_LABELS = [f"outer_ray_{int(a):03d}deg_um"     for a in RAY_ANGLES]
INNER_RAY_COL_LABELS = [f"inner_ray_{int(a):03d}deg_um"     for a in RAY_ANGLES]
THICK_RAY_COL_LABELS = [f"thickness_ray_{int(a):03d}deg_um" for a in RAY_ANGLES]


# --------------------------
# Helpers
# --------------------------
def darken_and_sharpen(image):
    darkened = cv2.convertScaleAbs(image, alpha=1.5, beta=-20)
    sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    return cv2.filter2D(darkened, -1, sharpen_kernel)


def build_tissue_mask(full_image):
    hsv = cv2.cvtColor(full_image, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(hsv)
    sat_mask       = cv2.inRange(S, 15, 255)
    not_too_bright = cv2.inRange(V, 0, 250)
    mask = cv2.bitwise_and(sat_mask, not_too_bright)
    k    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    return mask


def compute_blur_mask_full_image(image, window_size=7, percentile=0):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray_denoised = cv2.GaussianBlur(gray, (3, 3), 0)
    lap = np.abs(cv2.Laplacian(gray_denoised, cv2.CV_64F))
    mean    = cv2.blur(lap, (window_size, window_size))
    mean_sq = cv2.blur(lap ** 2, (window_size, window_size))
    var     = mean_sq - mean ** 2
    var     = cv2.normalize(var, None, 0, 255, cv2.NORM_MINMAX)
    blur_map = var.astype(np.uint8)
    thresh   = np.percentile(blur_map, percentile)
    blur_mask = np.zeros_like(blur_map, dtype=np.uint8)
    blur_mask[blur_map <= thresh] = 255   # white = blurred
    return blur_map, blur_mask


def contour_overlap_ratio(contour, mask):
    c_mask = np.zeros(mask.shape, dtype=np.uint8)
    cv2.drawContours(c_mask, [contour], -1, 255, -1)
    inter  = cv2.bitwise_and(c_mask, mask)
    area_c = cv2.countNonZero(c_mask)
    area_i = cv2.countNonZero(inter)
    return area_i / float(area_c) if area_c > 0 else 0.0


def circularity(contour):
    a = cv2.contourArea(contour)
    p = cv2.arcLength(contour, True)
    return 4 * np.pi * a / (p * p + 1e-6) if p > 0 else 0


# --------------------------
# Ray casting
# --------------------------
def cast_rays_to_boundary(contour, cx, cy, n_rays=N_RAYS, step=RAY_STEP):
    x_b, y_b, w_b, h_b = cv2.boundingRect(contour)
    pad      = 20
    max_r    = int(max(w_b, h_b) + pad)
    local_cx = max_r
    local_cy = max_r
    shifted  = contour - np.array([[int(cx) - max_r, int(cy) - max_r]])

    mask_size  = max_r * 2 + 2
    local_mask = np.zeros((mask_size, mask_size), dtype=np.uint8)
    cv2.drawContours(local_mask, [shifted], -1, 255, -1)

    angles           = np.linspace(0, 2 * np.pi, n_rays, endpoint=False)
    ray_distances_px = np.zeros(n_rays)

    for i, angle in enumerate(angles):
        cos_a  = np.cos(angle)
        sin_a  = np.sin(angle)
        r_vals = np.arange(0, max_r, step)
        xs     = (local_cx + r_vals * cos_a).astype(int)
        ys     = (local_cy + r_vals * sin_a).astype(int)
        valid  = (xs >= 0) & (xs < mask_size) & (ys >= 0) & (ys < mask_size)
        xs_v, ys_v, r_v = xs[valid], ys[valid], r_vals[valid]

        if len(xs_v) == 0:
            continue

        pixel_vals  = local_mask[ys_v, xs_v]
        inside_idx  = np.where(pixel_vals == 255)[0]
        if len(inside_idx) == 0:
            continue

        outside_after = np.where(
            (pixel_vals == 0) & (np.arange(len(pixel_vals)) > inside_idx[0])
        )[0]
        ray_distances_px[i] = r_v[outside_after[0]] if len(outside_after) else r_v[-1]

    return ray_distances_px, np.degrees(angles)


def draw_rays_on_image(output_image, cx, cy, ray_distances_px,
                       ray_angles_deg, chosen_idx, x_offset, y_offset):
    angles_rad = np.deg2rad(ray_angles_deg)
    for i, (r, ang) in enumerate(zip(ray_distances_px, angles_rad)):
        end_x = int(cx + r * np.cos(ang))
        end_y = int(cy + r * np.sin(ang))
        color = (0, 0, 255) if i == chosen_idx else (200, 200, 0)
        cv2.line(output_image,
                 (int(cx + x_offset), int(cy + y_offset)),
                 (end_x + x_offset,   end_y + y_offset),
                 color, 1)


# --------------------------
# Patch processing — NO blur mask, detects everything
# --------------------------
def process_patch(patch, mask_patch, x_offset, y_offset,
                  axon_data, object_counter, output_image, axons_image):

    patch    = darken_and_sharpen(patch)
    gray     = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    smooth   = cv2.bilateralFilter(gray, 5, 20, 20)
    clahe    = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(smooth)
    blurred  = cv2.GaussianBlur(enhanced, (3, 3), 0)

    _, otsu  = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    adaptive = cv2.adaptiveThreshold(blurred, 255,
                                     cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY_INV, 35, 2)
    thresh   = cv2.bitwise_or(otsu, adaptive)

    kernel  = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    sure_bg = cv2.dilate(opening, kernel, iterations=3)

    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    if dist_transform.max() > 0:
        _, sure_fg = cv2.threshold(dist_transform, 0.4 * dist_transform.max(), 255, 0)
    else:
        sure_fg = np.zeros_like(opening)
    sure_fg = np.uint8(sure_fg)

    unknown = cv2.subtract(sure_bg, sure_fg)
    num_markers, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0

    wshed_input = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    markers     = cv2.watershed(wshed_input, markers)

    cleaned = thresh.copy()
    cleaned[markers == -1] = 0

    contours, hierarchy = cv2.findContours(cleaned, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    if hierarchy is None:
        return object_counter
    hierarchy = hierarchy[0]

    for i, contour in enumerate(contours):
        if hierarchy[i][3] != -1:
            continue

        area = cv2.contourArea(contour)
        if area < MIN_CONTOUR_AREA or area > MAX_CONTOUR_AREA:
            continue

        overlap = contour_overlap_ratio(contour, mask_patch)
        if overlap < TISSUE_OVERLAP_MIN:
            continue

        hull     = cv2.convexHull(contour)
        solidity = area / (cv2.contourArea(hull) + 1e-6)
        circ     = circularity(contour)
        if solidity < MIN_SOLIDITY or circ < MIN_CIRCULARITY:
            continue

        inner_children = [
            contours[j] for j in range(len(contours))
            if hierarchy[j][3] == i and cv2.contourArea(contours[j]) > 30
        ]
        if len(inner_children) == 0:
            continue

        axon_type = "mature" if len(inner_children) <= 2 else "regrowth_cluster"

        (x_outer, y_outer), outer_radius_enc = cv2.minEnclosingCircle(contour)
        contour_shifted = contour + np.array([x_offset, y_offset])
        x_outer_abs     = x_outer + x_offset
        y_outer_abs     = y_outer + y_offset

        outer_area_px  = cv2.contourArea(contour)
        outer_area_um2 = outer_area_px * (scale_factor ** 2)
        outer_color    = (255, 0, 0) if axon_type == "mature" else (0, 255, 255)

        cv2.drawContours(output_image, [contour_shifted], -1, outer_color, 1)
        cv2.drawContours(axons_image,  [contour_shifted], -1, outer_color, 1)

        inner_radii, inner_areas = [], []
        for inner_c in inner_children:
            (_, _), ir = cv2.minEnclosingCircle(inner_c)
            inner_shifted = inner_c + np.array([x_offset, y_offset])
            cv2.drawContours(output_image, [inner_shifted], -1, (0, 255, 0), 1)
            cv2.drawContours(axons_image,  [inner_shifted], -1, (0, 255, 0), 1)
            inner_radii.append(ir)
            inner_areas.append(cv2.contourArea(inner_c))

        inner_radius   = max(inner_radii)
        inner_area_px  = max(inner_areas)
        inner_area_um2 = inner_area_px * (scale_factor ** 2)

        outer_ray_dict = {col: np.nan for col in OUTER_RAY_COL_LABELS}
        inner_ray_dict = {col: np.nan for col in INNER_RAY_COL_LABELS}
        thick_ray_dict = {col: np.nan for col in THICK_RAY_COL_LABELS}

        outer_radius_um  = outer_radius_enc * scale_factor
        inner_radius_um  = inner_radius     * scale_factor
        chosen_outer_um  = outer_radius_um
        chosen_inner_um  = inner_radius_um
        chosen_thick_um  = max(outer_radius_um - inner_radius_um, 0.0)
        chosen_ray_angle = np.nan
        chosen_idx       = -1

        if axon_type == "mature":
            outer_rays_px, ray_angles_deg = cast_rays_to_boundary(contour, x_outer, y_outer)
            outer_rays_um = outer_rays_px * scale_factor

            largest_inner_c  = inner_children[np.argmax(inner_areas)]
            inner_rays_px, _ = cast_rays_to_boundary(largest_inner_c, x_outer, y_outer)
            inner_rays_um    = inner_rays_px * scale_factor
            thick_per_dir    = outer_rays_um - inner_rays_um

            for o_col, i_col, t_col, o_val, i_val, t_val in zip(
                OUTER_RAY_COL_LABELS, INNER_RAY_COL_LABELS, THICK_RAY_COL_LABELS,
                outer_rays_um, inner_rays_um, thick_per_dir
            ):
                outer_ray_dict[o_col] = round(float(o_val), 4)
                inner_ray_dict[i_col] = round(float(i_val), 4)
                thick_ray_dict[t_col] = round(float(t_val), 4)

            positive_mask = thick_per_dir > 0
            if positive_mask.any():
                valid_thick      = np.where(positive_mask, thick_per_dir, np.inf)
                chosen_idx       = int(np.argmin(valid_thick))
                chosen_outer_um  = float(outer_rays_um[chosen_idx])
                chosen_inner_um  = float(inner_rays_um[chosen_idx])
                chosen_thick_um  = float(thick_per_dir[chosen_idx])
                chosen_ray_angle = float(ray_angles_deg[chosen_idx])
            else:
                chosen_thick_um  = max(outer_radius_um - inner_radius_um, 0.0)

            draw_rays_on_image(output_image, x_outer, y_outer,
                               outer_rays_px, ray_angles_deg,
                               chosen_idx, x_offset, y_offset)

        thickness_um = chosen_thick_um
        g_ratio      = chosen_inner_um / (chosen_outer_um + 1e-6)
        area_ratio   = inner_area_um2  / (outer_area_um2  + 1e-6)

        record = {
            "axon_id"              : object_counter,
            "axon_type"            : axon_type,
            "num_inner_contours"   : len(inner_children),
            "center_x_px"          : round(float(x_outer_abs), 2),
            "center_y_px"          : round(float(y_outer_abs), 2),
            "outer_radius_enc_um"  : round(outer_radius_um,  4),
            "inner_radius_enc_um"  : round(inner_radius_um,  4),
            "chosen_ray_angle_deg" : round(chosen_ray_angle, 2) if axon_type == "mature" else np.nan,
            "chosen_outer_ray_um"  : round(chosen_outer_um,  4),
            "chosen_inner_ray_um"  : round(chosen_inner_um,  4),
            "thickness_um"         : round(thickness_um,     4),
            "diameter_um"          : round(2 * chosen_outer_um, 4),
            "outer_area_um2"       : round(outer_area_um2,   4),
            "inner_area_um2"       : round(inner_area_um2,   4),
            "area_ratio"           : round(area_ratio,        4),
            "g_ratio"              : round(g_ratio,           4),
            "contour"              : contour_shifted.copy(),   # needed for blur filter
            "inner_contours" : [ic + np.array([x_offset, y_offset]) for ic in inner_children],
        }
        record.update(outer_ray_dict)
        record.update(inner_ray_dict)
        record.update(thick_ray_dict)
        axon_data.append(record)

        cv2.putText(output_image, str(object_counter),
                    (int(x_outer_abs), int(y_outer_abs)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        cv2.putText(axons_image, str(object_counter),
                    (int(x_outer_abs), int(y_outer_abs)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        object_counter += 1

    return object_counter


# --------------------------
# Whole-image processing
# --------------------------
def process_image(image_path, patch_size=1024):
    image = cv2.imread(image_path)
    if image is None:
        print(f"❌ Cannot read: {image_path}")
        return None

    tissue_mask_full = build_tissue_mask(image)
    h, w = image.shape[:2]

    output_image   = image.copy()
    axons_image    = image.copy()
    axon_data      = []
    object_counter = 1

    for y in range(0, h, patch_size):
        for x in range(0, w, patch_size):
            patch      = image[y:y + patch_size, x:x + patch_size]
            mask_patch = tissue_mask_full[y:y + patch_size, x:x + patch_size]
            object_counter = process_patch(
                patch, mask_patch, x, y,
                axon_data, object_counter,
                output_image, axons_image
            )

    print(f"  Detected {len(axon_data)} axons total.")

    # --------------------------
    # Post-detection blur filter (80% overlap rule)
    # --------------------------
    blur_map, blur_mask = compute_blur_mask_full_image(image, window_size=7, percentile=0)

    filtered_axons = []
    removed_axons  = []

    for axon in axon_data:
        contour      = axon["contour"]
        c_mask       = np.zeros(blur_mask.shape, dtype=np.uint8)
        cv2.drawContours(c_mask, [contour], -1, 255, -1)
        overlap      = cv2.bitwise_and(c_mask, blur_mask)
        overlap_area = cv2.countNonZero(overlap)
        contour_area = cv2.countNonZero(c_mask)

        if overlap_area >= 0.80 * contour_area:
            removed_axons.append(axon)
        else:
            filtered_axons.append(axon)

    print(f"  Before blur filter : {len(axon_data)}")
    print(f"  Removed (≥80% blur): {len(removed_axons)}")
    print(f"  After blur filter  : {len(filtered_axons)}")

    # before image — all detections
    output_before = image.copy()
    for axon in axon_data:
        outer_color = (255, 0, 0) if axon["axon_type"] == "mature" else (0, 255, 255)
        cv2.drawContours(output_before, [axon["contour"]], -1, outer_color, 1)
        for ic in axon["inner_contours"]:
            cv2.drawContours(output_before, [ic], -1, (0, 255, 0), 1)

    output_after = image.copy()
    for axon in filtered_axons:
        outer_color = (255, 0, 0) if axon["axon_type"] == "mature" else (0, 255, 255)
        cv2.drawContours(output_after, [axon["contour"]], -1, outer_color, 1)
        for ic in axon["inner_contours"]:
            cv2.drawContours(output_after, [ic], -1, (0, 255, 0), 1)

    return filtered_axons, output_before, output_after, blur_map, blur_mask


# --------------------------
# Excel export
# --------------------------
def save_to_excel(axon_data, output_folder, image_name):
    if not axon_data:
        print("  ⚠ No axon data to save.")
        return None

    df = pd.DataFrame(axon_data)

    # drop contour column before saving
    if "contour" in df.columns:
        df = df.drop(columns=["contour"])

    base_cols = [
        "axon_id", "axon_type", "num_inner_contours",
        "center_x_px", "center_y_px",
        "outer_radius_enc_um", "inner_radius_enc_um",
        *OUTER_RAY_COL_LABELS, *INNER_RAY_COL_LABELS, *THICK_RAY_COL_LABELS,
        "chosen_ray_angle_deg", "chosen_outer_ray_um", "chosen_inner_ray_um",
        "thickness_um", "diameter_um",
        "outer_area_um2", "inner_area_um2",
        "area_ratio", "g_ratio",
    ]
    ordered_cols = [c for c in base_cols if c in df.columns]
    df = df[ordered_cols]

    excel_path = os.path.join(output_folder, f"{image_name}_axon_measurements.xlsx")
    df.to_excel(excel_path, index=False, sheet_name="Axon Measurements")

    wb = openpyxl.load_workbook(excel_path)
    ws = wb.active

    HEADER_FILL    = PatternFill("solid", fgColor="1F4E79")
    OUTER_RAY_FILL = PatternFill("solid", fgColor="E8F4FD")
    INNER_RAY_FILL = PatternFill("solid", fgColor="EAF4E8")
    THICK_RAY_FILL = PatternFill("solid", fgColor="FFF2CC")
    MATURE_FILL    = PatternFill("solid", fgColor="D9F0D3")
    REGROWTH_FILL  = PatternFill("solid", fgColor="FCE4D6")
    CHOSEN_FILL    = PatternFill("solid", fgColor="FFD700")
    THICK_FILL     = PatternFill("solid", fgColor="F4CCCC")
    WHITE_FONT     = Font(color="FFFFFF", bold=True, name="Calibri", size=10)
    HEADER_BORDER  = Border(bottom=Side(style="medium", color="FFFFFF"))
    CELL_BORDER    = Border(bottom=Side(style="thin",   color="D0D0D0"))

    col_names = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]

    def col_idx(name):
        try:    return col_names.index(name) + 1
        except: return None

    outer_ray_col_indices = [col_idx(c) for c in OUTER_RAY_COL_LABELS if col_idx(c)]
    inner_ray_col_indices = [col_idx(c) for c in INNER_RAY_COL_LABELS if col_idx(c)]
    thick_ray_col_indices = [col_idx(c) for c in THICK_RAY_COL_LABELS if col_idx(c)]
    chosen_cols           = {col_idx(c) for c in
                             ("chosen_ray_angle_deg", "chosen_outer_ray_um", "chosen_inner_ray_um")
                             if col_idx(c)}
    thickness_col  = col_idx("thickness_um")
    axon_type_col  = col_idx("axon_type")

    for cell in ws[1]:
        cell.fill      = HEADER_FILL
        cell.font      = WHITE_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = HEADER_BORDER

    for ci in outer_ray_col_indices:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="1565C0")
    for ci in inner_ray_col_indices:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="2E7D32")
    for ci in thick_ray_col_indices:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="F57F17")
    for ci in chosen_cols:
        ws.cell(1, ci).fill = PatternFill("solid", fgColor="B8860B")
    if thickness_col:
        ws.cell(1, thickness_col).fill = PatternFill("solid", fgColor="8B0000")
        ws.cell(1, thickness_col).font = Font(color="FFFFFF", bold=True, name="Calibri", size=10)

    for row_idx in range(2, ws.max_row + 1):
        atype    = ws.cell(row_idx, axon_type_col).value if axon_type_col else ""
        row_fill = MATURE_FILL if atype == "mature" else REGROWTH_FILL

        for col_idx_ in range(1, ws.max_column + 1):
            cell           = ws.cell(row_idx, col_idx_)
            cell.border    = CELL_BORDER
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.fill      = row_fill

            if   col_idx_ in outer_ray_col_indices: cell.fill = OUTER_RAY_FILL
            elif col_idx_ in inner_ray_col_indices: cell.fill = INNER_RAY_FILL
            elif col_idx_ in thick_ray_col_indices: cell.fill = THICK_RAY_FILL
            elif col_idx_ in chosen_cols:
                cell.fill = CHOSEN_FILL
                cell.font = Font(bold=True, name="Calibri", size=10)
            elif thickness_col and col_idx_ == thickness_col:
                cell.fill = THICK_FILL
                cell.font = Font(bold=True, name="Calibri", size=10)

    for col_num in range(1, ws.max_column + 1):
        col_letter = get_column_letter(col_num)
        header_val = str(ws.cell(1, col_num).value or "")
        if header_val.startswith(("outer_ray_", "inner_ray_", "thickness_ray_")):
            ws.column_dimensions[col_letter].width = 14
        elif header_val in ("axon_id", "axon_type", "num_inner_contours"):
            ws.column_dimensions[col_letter].width = 16
        else:
            ws.column_dimensions[col_letter].width = 20

    ws.row_dimensions[1].height = 40
    ws.freeze_panes = "A2"

    ws_summary  = wb.create_sheet("Summary")
    mature_df   = df[df["axon_type"] == "mature"]
    regrowth_df = df[df["axon_type"] == "regrowth_cluster"]

    summary_rows = [
        ("Image",                       image_name),
        ("Total axons",                 len(df)),
        ("Mature axons",                len(mature_df)),
        ("Regrowth clusters",           len(regrowth_df)),
        ("", ""),
        ("--- Mature Axons ---", ""),
        ("Mean outer radius enc. (µm)", round(mature_df["outer_radius_enc_um"].mean(), 4) if len(mature_df) else "N/A"),
        ("Mean inner radius enc. (µm)", round(mature_df["inner_radius_enc_um"].mean(), 4) if len(mature_df) else "N/A"),
        ("Mean chosen outer ray (µm)",  round(mature_df["chosen_outer_ray_um"].mean(),  4) if len(mature_df) else "N/A"),
        ("Mean chosen inner ray (µm)",  round(mature_df["chosen_inner_ray_um"].mean(),  4) if len(mature_df) else "N/A"),
        ("Mean thickness (µm)",         round(mature_df["thickness_um"].mean(),          4) if len(mature_df) else "N/A"),
        ("Mean g-ratio",                round(mature_df["g_ratio"].mean(),               4) if len(mature_df) else "N/A"),
        ("Mean diameter (µm)",          round(mature_df["diameter_um"].mean(),           4) if len(mature_df) else "N/A"),
        ("", ""),
        ("--- Regrowth Clusters ---", ""),
        ("Mean outer radius enc. (µm)", round(regrowth_df["outer_radius_enc_um"].mean(), 4) if len(regrowth_df) else "N/A"),
        ("Mean thickness (µm)",         round(regrowth_df["thickness_um"].mean(),        4) if len(regrowth_df) else "N/A"),
    ]

    for r, (label, value) in enumerate(summary_rows, start=1):
        c1 = ws_summary.cell(r, 1, value=label)
        c2 = ws_summary.cell(r, 2, value=value)
        c1.font      = Font(bold=True, name="Calibri", size=10)
        c2.alignment = Alignment(horizontal="center")
        if str(label).startswith("---"):
            c1.fill = HEADER_FILL
            c1.font = WHITE_FONT

    ws_summary.column_dimensions["A"].width = 38
    ws_summary.column_dimensions["B"].width = 20
    wb.save(excel_path)
    print(f"  ✔ Excel saved → {excel_path}")
    return df


# --------------------------
# Process folder
# --------------------------
def process_folder(input_folder, output_root):
    images = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
    ]
    if not images:
        print("⚠ No images found.")
        return

    for img_name in images:
        print(f"\n{'='*30}\n Processing: {img_name}\n{'='*30}")
        img_path          = os.path.join(input_folder, img_name)
        img_name_wo_ext   = os.path.splitext(img_name)[0]
        img_output_folder = os.path.join(output_root, img_name_wo_ext)
        os.makedirs(img_output_folder, exist_ok=True)

        result = process_image(img_path)
        if result is None:
            continue

        filtered_axons, output_before, output_after, blur_map, blur_mask = result

        save_to_excel(filtered_axons, img_output_folder, img_name_wo_ext)

        cv2.imwrite(os.path.join(img_output_folder, f"01_before_{img_name}"), output_before)
        cv2.imwrite(os.path.join(img_output_folder, f"02_after_{img_name}"),  output_after)
        cv2.imwrite(os.path.join(img_output_folder, f"03_blur_map_{img_name}"),  blur_map)
        cv2.imwrite(os.path.join(img_output_folder, f"04_blur_mask_{img_name}"), blur_mask)
        print(f"  ✔ Saved → {img_output_folder}")

    print("\n🎉 Done.")


process_folder(input_folder, output_root)


 Processing: 1.png
  Detected 190 axons total.
  Before blur filter : 190
  Removed (≥80% blur): 1
  After blur filter  : 189
  ✔ Excel saved → C:\Users\DELL\Downloads\20x_bf_02_detection\1\1_axon_measurements.xlsx
  ✔ Saved → C:\Users\DELL\Downloads\20x_bf_02_detection\1

 Processing: 2.png
  Detected 12 axons total.
  Before blur filter : 12
  Removed (≥80% blur): 0
  After blur filter  : 12
  ✔ Excel saved → C:\Users\DELL\Downloads\20x_bf_02_detection\2\2_axon_measurements.xlsx
  ✔ Saved → C:\Users\DELL\Downloads\20x_bf_02_detection\2

 Processing: 3.png
  Detected 406 axons total.
  Before blur filter : 406
  Removed (≥80% blur): 8
  After blur filter  : 398
  ✔ Excel saved → C:\Users\DELL\Downloads\20x_bf_02_detection\3\3_axon_measurements.xlsx
  ✔ Saved → C:\Users\DELL\Downloads\20x_bf_02_detection\3

 Processing: 4.png
  Detected 726 axons total.
  Before blur filter : 726
  Removed (≥80% blur): 2
  After blur filter  : 724
  ✔ Excel saved → C:\Users\DELL\Downloads\20x_bf_02_de